In [1]:
import pandas as pd

df = pd.read_csv('nassau_candy_clean.csv')
print(df.shape)
print(df.columns.tolist())

(10194, 22)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost', 'Lead Time Raw', 'Lead Time (Days)', 'Factory', 'Route']


In [2]:
route_stats = df.groupby('Route').agg(
    Total_Shipments  = ('Lead Time (Days)', 'count'),
    Avg_Lead_Time    = ('Lead Time (Days)', 'mean'),
    Median_Lead_Time = ('Lead Time (Days)', 'median'),
    Std_Lead_Time    = ('Lead Time (Days)', 'std'),
    Min_Lead_Time    = ('Lead Time (Days)', 'min'),
    Max_Lead_Time    = ('Lead Time (Days)', 'max'),
).reset_index()

route_stats = route_stats.round(2)
print(route_stats.shape)
print(route_stats.head(10))

(196, 7)
                                  Route  Total_Shipments  Avg_Lead_Time  \
0               Lot's O' Nuts → Alabama               34           4.74   
1               Lot's O' Nuts → Alberta               16           4.88   
2               Lot's O' Nuts → Arizona              111           4.32   
3              Lot's O' Nuts → Arkansas               31           4.58   
4      Lot's O' Nuts → British Columbia               18           4.78   
5            Lot's O' Nuts → California             1125           4.11   
6              Lot's O' Nuts → Colorado              103           4.12   
7           Lot's O' Nuts → Connecticut               47           4.00   
8              Lot's O' Nuts → Delaware               43           4.30   
9  Lot's O' Nuts → District of Columbia                8           5.62   

   Median_Lead_Time  Std_Lead_Time  Min_Lead_Time  Max_Lead_Time  
0               5.0           1.54              2              8  
1               5.0           0

In [3]:
DELAY_THRESHOLD = 7

df['Is_Delayed'] = (df['Lead Time (Days)'] > DELAY_THRESHOLD).astype(int)

delay_rate = df.groupby('Route')['Is_Delayed'].mean().reset_index()
delay_rate.columns = ['Route', 'Delay_Rate']
delay_rate['Delay_Rate'] = (delay_rate['Delay_Rate'] * 100).round(2)

route_stats = route_stats.merge(delay_rate, on='Route')
print(route_stats[['Route','Total_Shipments','Avg_Lead_Time','Delay_Rate']].head(10))

                                  Route  Total_Shipments  Avg_Lead_Time  \
0               Lot's O' Nuts → Alabama               34           4.74   
1               Lot's O' Nuts → Alberta               16           4.88   
2               Lot's O' Nuts → Arizona              111           4.32   
3              Lot's O' Nuts → Arkansas               31           4.58   
4      Lot's O' Nuts → British Columbia               18           4.78   
5            Lot's O' Nuts → California             1125           4.11   
6              Lot's O' Nuts → Colorado              103           4.12   
7           Lot's O' Nuts → Connecticut               47           4.00   
8              Lot's O' Nuts → Delaware               43           4.30   
9  Lot's O' Nuts → District of Columbia                8           5.62   

   Delay_Rate  
0        2.94  
1        0.00  
2        1.80  
3        6.45  
4        0.00  
5        1.42  
6        0.97  
7        0.00  
8        4.65  
9        0.00 

In [4]:
min_lt = route_stats['Avg_Lead_Time'].min()
max_lt = route_stats['Avg_Lead_Time'].max()

route_stats['Efficiency_Score'] = (
    (max_lt - route_stats['Avg_Lead_Time']) / (max_lt - min_lt) * 100
).round(2)

print(route_stats[['Route','Avg_Lead_Time','Efficiency_Score']]
      .sort_values('Efficiency_Score', ascending=False)
      .head(10))

                                  Route  Avg_Lead_Time  Efficiency_Score
129  The Other Factory → North Carolina            1.0            100.00
76            Secret Factory → Nebraska            2.0             85.71
80          Secret Factory → New Mexico            2.0             85.71
120        The Other Factory → Kentucky            2.0             85.71
126   The Other Factory → New Hampshire            2.0             85.71
131          The Other Factory → Oregon            2.5             78.57
101              Sugar Shack → Illinois            2.5             78.57
84              Secret Factory → Oregon            2.5             78.57
20             Lot's O' Nuts → Manitoba            2.5             78.57
112         The Other Factory → Arizona            2.5             78.57


In [5]:
top10 = route_stats.nlargest(10, 'Efficiency_Score')[
    ['Route','Avg_Lead_Time','Total_Shipments','Efficiency_Score']
]
bottom10 = route_stats.nsmallest(10, 'Efficiency_Score')[
    ['Route','Avg_Lead_Time','Total_Shipments','Efficiency_Score']
]

print("=== TOP 10 MOST EFFICIENT ROUTES ===")
print(top10.to_string(index=False))
print()
print("=== BOTTOM 10 LEAST EFFICIENT ROUTES ===")
print(bottom10.to_string(index=False))

=== TOP 10 MOST EFFICIENT ROUTES ===
                             Route  Avg_Lead_Time  Total_Shipments  Efficiency_Score
The Other Factory → North Carolina            1.0                1            100.00
         Secret Factory → Nebraska            2.0                1             85.71
       Secret Factory → New Mexico            2.0                2             85.71
      The Other Factory → Kentucky            2.0                2             85.71
 The Other Factory → New Hampshire            2.0                2             85.71
          Lot's O' Nuts → Manitoba            2.5                2             78.57
           Secret Factory → Oregon            2.5                2             78.57
            Sugar Shack → Illinois            2.5                2             78.57
       The Other Factory → Arizona            2.5                2             78.57
        The Other Factory → Oregon            2.5                2             78.57

=== BOTTOM 10 LEAST EFFICIE

In [6]:
region_stats = df.groupby('Region').agg(
    Total_Shipments = ('Lead Time (Days)', 'count'),
    Avg_Lead_Time   = ('Lead Time (Days)', 'mean'),
    Delay_Rate      = ('Is_Delayed', 'mean'),
).reset_index().round(2)
region_stats['Delay_Rate'] = (region_stats['Delay_Rate'] * 100).round(2)
print(region_stats)

print()

shipmode_stats = df.groupby('Ship Mode').agg(
    Total_Shipments = ('Lead Time (Days)', 'count'),
    Avg_Lead_Time   = ('Lead Time (Days)', 'mean'),
    Delay_Rate      = ('Is_Delayed', 'mean'),
).reset_index().round(2)
shipmode_stats['Delay_Rate'] = (shipmode_stats['Delay_Rate'] * 100).round(2)
print(shipmode_stats)

     Region  Total_Shipments  Avg_Lead_Time  Delay_Rate
0  Atlantic             2986           4.24         2.0
1      Gulf             1620           4.30         2.0
2  Interior             2335           4.38         2.0
3   Pacific             3253           4.27         2.0

        Ship Mode  Total_Shipments  Avg_Lead_Time  Delay_Rate
0     First Class             1548           2.55         0.0
1        Same Day              547           0.38         0.0
2    Second Class             1979           3.57         0.0
3  Standard Class             6120           5.32         4.0


In [7]:
route_stats.to_csv('route_stats.csv', index=False)
region_stats.to_csv('region_stats.csv', index=False)
shipmode_stats.to_csv('shipmode_stats.csv', index=False)

print("✅ Saved: route_stats.csv")
print("✅ Saved: region_stats.csv")
print("✅ Saved: shipmode_stats.csv")
print()
print("route_stats columns:", route_stats.columns.tolist())
print("Total routes:", len(route_stats))

✅ Saved: route_stats.csv
✅ Saved: region_stats.csv
✅ Saved: shipmode_stats.csv

route_stats columns: ['Route', 'Total_Shipments', 'Avg_Lead_Time', 'Median_Lead_Time', 'Std_Lead_Time', 'Min_Lead_Time', 'Max_Lead_Time', 'Delay_Rate', 'Efficiency_Score']
Total routes: 196
